# Dense-эмбеддинги (семантика)

Одним взглядом на [статью](https://habr.com/ru/articles/669674/)решил, что на макбук беру эмбедер `rubert-tiny2`, `sentence-transformers` для инференса, `faiss-cpu` для ANN поиска.

Продолжение [baseline.ipynb](baseline.ipynb) вынес в отдельный ноутбук, чтобы не мешать логику эмбеддингов с lookup/BM25.

In [1]:
import os
import shutil
import sys
import tempfile
from collections import defaultdict

sys.path.append("..")

import bm25s
import faiss
import gdown
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from eval import DEFAULT_SEARCH_COLS, make_val_split, recall_at_50
from preprocessing import clean_text, tokenize

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)

print("train_part:", len(train_part), "строк")
print("val_queries:", len(val_queries), "запросов")

train_part: 448040 строк
val_queries: 35447 запросов


## Семантика

Модель - `cointegrated/rubert-tiny2` (по [бенчам](https://habr.com/ru/articles/669674/)) - лучший trade-off.\
На вход - `clean_text()` (только фикс гомоглифов, без лемм). Локально на маке (24 ГБ RAM) даже `max_seq_length=512` постоянно ловил MPS OOM - поэтому корпус посчитан на **Kaggle GPU** (`kaggle_kernel/embed.py`) с `max_seq_length=1024`, результат скачан и лежит как обычный кэш или, если Вы это не сделали, **скачайте**!!.

In [2]:
EMB_CACHE = "../data/embeddings/rubert_tiny2/1024.npy"
EMB_IDS_CACHE = "../data/embeddings/rubert_tiny2/1024_item_ids.npy"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

mps


In [3]:
embed_model = SentenceTransformer("cointegrated/rubert-tiny2", device=device)
embed_model.max_seq_length = 1024

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
coverage_sample = train.drop_duplicates("item_id").sample(2000, random_state=42)
coverage_texts = (coverage_sample["item_title_raw"] + ". " + coverage_sample["item_description_raw"]).tolist()

_orig_max_len = embed_model.tokenizer.model_max_length
embed_model.tokenizer.model_max_length = int(1e9)  # снимаем лимит только для замера длины
token_lengths = np.array([len(embed_model.tokenizer.encode(t)) for t in coverage_texts])
embed_model.tokenizer.model_max_length = _orig_max_len

print("медиана токенов:", int(np.median(token_lengths)))
for thresh in [256, 512, 1024, 2048]:
    coverage = (token_lengths <= thresh).mean() * 100
    print(f"покрытие при max_seq_length={thresh}: {coverage:.1f}%")

медиана токенов: 214
покрытие при max_seq_length=256: 56.1%
покрытие при max_seq_length=512: 82.2%
покрытие при max_seq_length=1024: 96.0%
покрытие при max_seq_length=2048: 99.8%


1024 даёт 96% покрытия vs 82% у 512 - но локально на этом маке (24 ГБ RAM) стабильно ловим MPS OOM независимо от batch_size/чанков (`other allocations` у системы держится на ~28 ГБ). Поэтому реальный расчёт корпуса ушёл на Kaggle GPU (см. `kaggle_kernel/`), а тут просто загружаем готовый результат из кэша.

In [5]:
dense_items = train.drop_duplicates("item_id")[
    ["item_id", "item_title_raw", "item_description_raw"]
].reset_index(drop=True)

DRIVE_FOLDER = "https://drive.google.com/drive/folders/1q_Xk0vMn0pBOwGNC3JnqQ__Y0ty-s0px"

if not (os.path.exists(EMB_CACHE) and os.path.exists(EMB_IDS_CACHE)):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            gdown.download_folder(DRIVE_FOLDER, output=tmp, quiet=False)
            src = f"{tmp}/embeddings"
            if os.path.exists(f"{src}/embeddings_rubert_tiny2_1024.npy"):
                shutil.copy(f"{src}/embeddings_rubert_tiny2_1024.npy", EMB_CACHE)
                shutil.copy(f"{src}/embeddings_rubert_tiny2_1024_item_ids.npy", EMB_IDS_CACHE)
                print("эмбеды скачаны с Google Drive")
    except Exception as e:  # noqa: BLE001
        print("не удалось скачать, посчитаем с нуля:", e)

if os.path.exists(EMB_CACHE):
    corpus_embeddings = np.load(EMB_CACHE)
    dense_item_ids = np.load(EMB_IDS_CACHE, allow_pickle=True).tolist()
    print("эмбеды из кэша:", corpus_embeddings.shape)
else:
    dense_items["text"] = (
        dense_items["item_title_raw"] + ". " + dense_items["item_description_raw"]
    ).apply(clean_text)

    CHUNK_SIZE = 5000
    SHARD_DIR = "../data/embeddings/shards"
    os.makedirs(SHARD_DIR, exist_ok=True)

    n_chunks = (len(dense_items) + CHUNK_SIZE - 1) // CHUNK_SIZE
    for i in range(n_chunks):
        shard_path = f"{SHARD_DIR}/shard_{i:04d}.npy"
        if os.path.exists(shard_path):
            continue
        chunk_texts = dense_items["text"].iloc[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE].tolist()
        chunk_emb = embed_model.encode(
            chunk_texts,
            batch_size=32,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        np.save(shard_path, chunk_emb)
        print(f"chunk {i + 1}/{n_chunks} готов")

    corpus_embeddings = np.concatenate(
        [np.load(f"{SHARD_DIR}/shard_{i:04d}.npy") for i in range(n_chunks)]
    )
    dense_item_ids = dense_items["item_id"].tolist()
    np.save(EMB_CACHE, corpus_embeddings)
    np.save(EMB_IDS_CACHE, np.array(dense_item_ids))
    print("эмбеды посчитаны и закэшированы:", corpus_embeddings.shape)

эмбеды из кэша: (344825, 312)


In [6]:
index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

In [7]:
val_query_texts = val_queries["search_query"].apply(clean_text).tolist()
query_embeddings = embed_model.encode(
    val_query_texts, batch_size=64, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=True,
)

_, indices = index.search(query_embeddings, 50)

dense_predictions = {
    cid: [dense_item_ids[i] for i in row if i != -1]
    for row, cid in zip(indices, val_queries["context_id"])
}

In [8]:
dense_recall = recall_at_50(dense_predictions, val_relevant)
print(f"эмбеды (rubert-tiny2) Recall@50: {dense_recall:.4f}")

эмбеды (rubert-tiny2) Recall@50: 0.0107


ОЧЕНЬ ПЛОХО! :(

## Union(lookup, BM25, dense)

Проверим: дают ли эмбеды **уникальный** вклад поверх lookup+BM25 - то, что лексика в принципе не видит - смысл, синонимы.

In [9]:
items_corpus = train.drop_duplicates("item_id")[
    ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text"]
].reset_index(drop=True)
items_corpus["text"] = (
    items_corpus["item_title_raw"] + " "
    + items_corpus["item_description_raw"] + " "
    + items_corpus["item_infm_params_text"]
)
corpus_item_ids_bm25 = items_corpus["item_id"].tolist()

BM25_CACHE = "../data/bm25s_index"
if os.path.exists(BM25_CACHE):
    retriever = bm25s.BM25.load(BM25_CACHE, load_corpus=False)
else:
    items_corpus["tokens"] = items_corpus["text"].apply(tokenize)
    retriever = bm25s.BM25()
    retriever.index(items_corpus["tokens"].tolist(), show_progress=False)
    retriever.save(BM25_CACHE)

val_tokens = val_queries["search_query"].apply(tokenize).tolist()
results, scores = retriever.retrieve(
    val_tokens, corpus=corpus_item_ids_bm25, k=50, show_progress=False, n_threads=0
)
bm25_predictions = {
    cid: list(results[i])
    for i, cid in enumerate(val_queries["context_id"])
}
bm25_recall = recall_at_50(bm25_predictions, val_relevant)
print(f"BM25 Recall@50: {bm25_recall:.4f}")

BM25 Recall@50: 0.1861


In [10]:
def build_lookup(item_ids, keys):
    lookup = defaultdict(dict)
    for key, item_id in zip(keys, item_ids):
        d = lookup[key]
        d[item_id] = d.get(item_id, 0) + 1
    return lookup


def rank_candidates(ctx, qtext, top_n=50, context_weight=2):
    scores = {}
    for item_id, cnt in context_lookup.get(ctx, {}).items():
        scores[item_id] = scores.get(item_id, 0) + cnt * context_weight
    for item_id, cnt in text_lookup.get(qtext, {}).items():
        scores[item_id] = scores.get(item_id, 0) + cnt
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [item_id for item_id, _ in ranked[:top_n]]


context_train = train_part[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1)
query_norm_train = train_part["search_query"].str.lower().str.strip()

context_lookup = build_lookup(train_part["item_id"], context_train)
text_lookup = build_lookup(train_part["item_id"], query_norm_train)

val_context = val_queries[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1)
val_query_norm = val_queries["search_query"].str.lower().str.strip()

lookup_predictions = {
    cid: rank_candidates(ctx, qtext)
    for cid, ctx, qtext in zip(val_queries["context_id"], val_context, val_query_norm)
}
lookup_recall = recall_at_50(lookup_predictions, val_relevant)
print(f"Lookup Recall@50: {lookup_recall:.4f}")

Lookup Recall@50: 0.0244


In [11]:
union3_predictions = {}
for cid in val_queries["context_id"]:
    combined = list(dict.fromkeys(
        list(bm25_predictions.get(cid, []))
        + list(lookup_predictions.get(cid, []))
        + list(dense_predictions.get(cid, []))
    ))[:50]
    union3_predictions[cid] = combined

union3_recall = recall_at_50(union3_predictions, val_relevant)
print(f"Union(lookup, BM25, dense) Recall@50: {union3_recall:.4f}")
print(f"BM25 отдельно: {bm25_recall:.4f}")
print(f"lookup отдельно: {lookup_recall:.4f}")
print(f"dense отдельно: {dense_recall:.4f}")
print(f"прирост над чистым BM25: {union3_recall - bm25_recall:+.4f}")

Union(lookup, BM25, dense) Recall@50: 0.1861
BM25 отдельно: 0.1861
lookup отдельно: 0.0244
dense отдельно: 0.0107
прирост над чистым BM25: +0.0000


### RRF - базовый способ пореранкать без обучения

Reciprocal Rank Fusion - не нужно обучение и сравнение разных шкал скоров.

In [12]:
def reciprocal_rank_fusion(*ranked_lists, k=60, top_n=50):
    scores = {}
    for ranked_list in ranked_lists:
        for rank, item_id in enumerate(ranked_list, start=1):
            scores[item_id] = scores.get(item_id, 0) + 1 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [item_id for item_id, _ in ranked[:top_n]]


rrf_predictions = {
    cid: reciprocal_rank_fusion(
        bm25_predictions.get(cid, []),
        lookup_predictions.get(cid, []),
        dense_predictions.get(cid, []),
    )
    for cid in val_queries["context_id"]
}
rrf_recall = recall_at_50(rrf_predictions, val_relevant)
print(f"RRF(lookup, BM25, dense) Recall@50: {rrf_recall:.4f}")
print(f"наивный union: {union3_recall:.4f}")

RRF(lookup, BM25, dense) Recall@50: 0.1284
наивный union: 0.1861


В одном из своих проектов (RAG + агент поверх файлов) ретрив был основан на multilingual-e5-base - работал замечательно, поэтому попробую его.

### `intfloat/multilingual-e5-base`

e5 требует префиксы `query: ` / `passage: ` на входе (запрос / документ) - без них качество проседает.

In [13]:
EMB_CACHE = "../data/embeddings/e5_base/512.npy"
EMB_IDS_CACHE = "../data/embeddings/e5_base/512_item_ids.npy"

In [14]:
embed_model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)
embed_model.max_seq_length = 512

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [15]:
if os.path.exists(EMB_CACHE):
    corpus_embeddings = np.load(EMB_CACHE)
    dense_item_ids = np.load(EMB_IDS_CACHE, allow_pickle=True).tolist()
    print("эмбеды из кэша:", corpus_embeddings.shape)
else:
    e5_texts = (
        "passage: " + dense_items["item_title_raw"] + ". " + dense_items["item_description_raw"]
    ).apply(clean_text)
    corpus_embeddings = embed_model.encode(
        e5_texts.tolist(), batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    dense_item_ids = dense_items["item_id"].tolist()
    np.save(EMB_CACHE, corpus_embeddings)
    np.save(EMB_IDS_CACHE, np.array(dense_item_ids))
    print("эмбеды посчитаны и закэшированы:", corpus_embeddings.shape)

эмбеды из кэша: (344825, 768)


In [16]:
index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

In [17]:
val_query_texts = val_queries["search_query"].apply(clean_text).apply(lambda t: f"query: {t}").tolist()
query_embeddings = embed_model.encode(
    val_query_texts, batch_size=64, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=True,
)

_, indices = index.search(query_embeddings, 50)

dense_predictions = {
    cid: [dense_item_ids[i] for i in row if i != -1]
    for row, cid in zip(indices, val_queries["context_id"])
}

In [18]:
dense_recall = recall_at_50(dense_predictions, val_relevant)
print(f"эмбеды (multilingual-e5-base) Recall@50: {dense_recall:.4f}")

эмбеды (multilingual-e5-base) Recall@50: 0.1786


## Union(lookup, BM25, dense=e5-base)

Тот же union/RRF, что и выше, но уже с рабочим dense-каналом (e5-base) вместо rubert-tiny2.

In [19]:
union_e5_predictions = {}
for cid in val_queries["context_id"]:
    combined = list(dict.fromkeys(
        list(bm25_predictions.get(cid, []))
        + list(lookup_predictions.get(cid, []))
        + list(dense_predictions.get(cid, []))
    ))[:50]
    union_e5_predictions[cid] = combined

union_e5_recall = recall_at_50(union_e5_predictions, val_relevant)
print(f"Union(lookup, BM25, dense=e5-base) Recall@50: {union_e5_recall:.4f}")
print(f"BM25 отдельно: {bm25_recall:.4f}")
print(f"lookup отдельно: {lookup_recall:.4f}")
print(f"dense (e5-base) отдельно: {dense_recall:.4f}")
print(f"прирост над чистым BM25: {union_e5_recall - bm25_recall:+.4f}")

Union(lookup, BM25, dense=e5-base) Recall@50: 0.1861
BM25 отдельно: 0.1861
lookup отдельно: 0.0244
dense (e5-base) отдельно: 0.1786
прирост над чистым BM25: +0.0000


### RRF с e5-base

In [20]:
rrf_e5_predictions = {
    cid: reciprocal_rank_fusion(
        bm25_predictions.get(cid, []),
        lookup_predictions.get(cid, []),
        dense_predictions.get(cid, []),
    )
    for cid in val_queries["context_id"]
}
rrf_e5_recall = recall_at_50(rrf_e5_predictions, val_relevant)
print(f"RRF(lookup, BM25, dense=e5-base) Recall@50: {rrf_e5_recall:.4f}")
print(f"наивный union: {union_e5_recall:.4f}")

RRF(lookup, BM25, dense=e5-base) Recall@50: 0.1883
наивный union: 0.1861


УРАААААА!\
RRF > union = bm25

Кажется, все равно это очень слабый сигнал от семантики, сравнимый с лексикой(\
 Надо использовать что-то еще помимо заголовка и описания, там вроде были параметры, надо попарсить их и добавить в эмбеды.